# Glow — Generative Flow with Invertible 1×1 Convolutions (2018)

_Papers · Generative Models_

**A normalizing flow whose channel-shuffling step is a learned, invertible 1×1 convolution — a generalization of a fixed channel permutation, with a cheap log-determinant — giving sharp, exact-likelihood image synthesis.**

---

This notebook is a practice scaffold for the **Glow — Generative Flow with Invertible 1×1 Convolutions (2018)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, math
torch.manual_seed(0)

# --- 0. Sanity-check the worked example: 1x1-conv log-det = h*w*log|det W| (Eq. 9). ---
W0 = torch.tensor([[2.0, 0.0], [1.0, 3.0]])     # det = 2*3 - 0*1 = 6
h, w = 4, 4
print("det(W) =", torch.det(W0).item())                         # 6.0
print("1x1-conv log-det h*w*log|det W| =",
      round((h * w * torch.log(torch.abs(torch.det(W0)))).item(), 4))   # 16*log6 ~ 28.6682
s_demo = torch.tensor([2.0, 0.5])               # actnorm scale: 2 * 0.5 = 1 -> volume preserved
print("actnorm log-det h*w*sum(log|s|) =",
      round((h * w * torch.log(torch.abs(s_demo)).sum()).item(), 4))    # 0.0


# --- 1. A tiny 4-channel toy "image" whose channels are correlated in TWO separate pairs. ---
# Fitting this needs the model to MIX channels, so the 1x1 conv matters (see the ablation).
C, H, Wd = 4, 4, 4
Mcorr = torch.tensor([[1.0, 0.9, 0.0, 0.0], [0.9, 1.0, 0.0, 0.0],
                      [0.0, 0.0, 1.0, -0.9], [0.0, 0.0, -0.9, 1.0]])
Lchol = torch.linalg.cholesky(Mcorr)            # mix independent factors into correlated channels
def sample_data(n):
    z = torch.randn(n, C, H, Wd)
    return torch.einsum('ij,njhw->nihw', Lchol, z)


# --- 2. The three flow components, BUILT BY HAND (Table 1). ---
class ActNorm(nn.Module):                       # y = exp(logs)*x + b ; log-det = h*w*sum(logs)
    def __init__(self, C):
        super().__init__()
        self.logs = nn.Parameter(torch.zeros(1, C, 1, 1))
        self.b    = nn.Parameter(torch.zeros(1, C, 1, 1))
        self.inited = False
    def forward(self, x):
        if not self.inited:                     # data-dependent init from the first batch
            with torch.no_grad():
                m = x.mean([0, 2, 3], keepdim=True)
                v = x.std([0, 2, 3], keepdim=True) + 1e-6
                self.b.data = -m / v; self.logs.data = -torch.log(v)
            self.inited = True
        return torch.exp(self.logs) * x + self.b, H * Wd * self.logs.sum()

class Inv1x1(nn.Module):                         # y_ij = W x_ij ; log-det = h*w*log|det W| (Eq. 9)
    def __init__(self, C):
        super().__init__()
        Q, _ = torch.linalg.qr(torch.randn(C, C))    # start from a rotation (det = +-1, invertible)
        self.W = nn.Parameter(Q)
    def forward(self, x):
        y = torch.einsum('ij,njhw->nihw', self.W, x)
        return y, H * Wd * torch.slogdet(self.W)[1]

class AffineCoupling(nn.Module):                 # split; (logs,t)=NN(xb); ya=exp(logs)*xa+t; yb=xb
    def __init__(self, C, hid=32):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(C // 2, hid, 3, padding=1), nn.ReLU(),
                                 nn.Conv2d(hid, hid, 1), nn.ReLU(),
                                 nn.Conv2d(hid, C, 3, padding=1))
        self.net[-1].weight.data.zero_(); self.net[-1].bias.data.zero_()   # start as identity
    def forward(self, x):
        xa, xb = x.chunk(2, 1)
        logs, t = self.net(xb).chunk(2, 1)
        logs = torch.tanh(logs)                  # stabilize the scale
        y = torch.cat([torch.exp(logs) * xa + t, xb], 1)
        return y, logs.sum([1, 2, 3])            # coupling log-det = sum(logs)


# --- 3. One step of flow = actnorm -> 1x1 conv -> coupling. use_1x1 toggles the ablation. ---
class Glow(nn.Module):
    def __init__(self, C, K=4, use_1x1=True):
        super().__init__()
        self.use_1x1 = use_1x1
        self.an = nn.ModuleList([ActNorm(C) for _ in range(K)])
        self.iv = nn.ModuleList([Inv1x1(C) for _ in range(K)])
        self.co = nn.ModuleList([AffineCoupling(C) for _ in range(K)])
    def forward(self, x):
        ld = 0
        for k in range(len(self.an)):
            x, l = self.an[k](x); ld = ld + l
            if self.use_1x1:
                x, l = self.iv[k](x); ld = ld + l    # ablation OFF -> channels never mix
            x, l = self.co[k](x); ld = ld + l
        logpz = (-0.5 * x ** 2 - 0.5 * math.log(2 * math.pi)).sum([1, 2, 3])
        return logpz + ld                        # exact log p(x) per sample (Eq. 7)

def bits_per_dim(model, x):                      # NLL / #values / log(2)  (lower is better)
    return (-model(x) / x[0].numel() / math.log(2)).mean()

def train(use_1x1=True, steps=1500):
    torch.manual_seed(0)
    m = Glow(C, K=4, use_1x1=use_1x1)
    opt = torch.optim.Adam(m.parameters(), 1e-3)
    for i in range(steps):
        loss = -m(sample_data(256)).mean() / (C * H * Wd)
        opt.zero_grad(); loss.backward(); opt.step()
        if i % 300 == 0 or i == steps - 1:
            with torch.no_grad():
                print(f"  step {i:4d}  bits/dim {bits_per_dim(m, sample_data(1000)).item():.4f}")
    return m

print("\nTRAIN with invertible 1x1 conv (channels mix):")
m_on = train(use_1x1=True)
print("\nABLATION: no 1x1 conv (channels never mix):")
m_off = train(use_1x1=False)

test = sample_data(2000)
with torch.no_grad():
    print(f"\nFINAL held-out bits/dim WITH 1x1 conv : {bits_per_dim(m_on, test).item():.4f}")
    print(f"FINAL held-out bits/dim NO   1x1 conv : {bits_per_dim(m_off, test).item():.4f}")
# WITH the learned 1x1 conv the channels mix and the model fits the correlation -> lower (better)
# bits/dim. WITHOUT it, one half is always passed through and the fit is worse.
# (Our small run -- not the paper's reported number.)

## Visualize the data & results

_Does the learned invertible 1×1 convolution actually help? Compare bits/dimension (lower is better) over training, with the 1×1 conv ON vs. ablated OFF._

In [ ]:
import torch, torch.nn as nn, math
torch.manual_seed(0)
# Same tiny Glow as the CODE cell; here we LOG held-out bits/dimension every 100 steps for both
# runs (1x1 conv ON vs. ablated OFF) and plot the two curves. Lower is better.
# Reproduces: ON settles ~1.45 bits/dim; OFF is stuck ~2.05 (the channels can never mix).
# (Numbers are our small run, not the paper's reported bits/dimension.)
#
# --- worked example (matches the lesson): 1x1-conv log-det = h*w*log|det W| ---
W0 = torch.tensor([[2.0, 0.0], [1.0, 3.0]]); h, w = 4, 4
print("det(W) =", torch.det(W0).item(),
      " h*w*log|det W| =", round((h*w*torch.log(torch.abs(torch.det(W0)))).item(), 4))  # 6.0  28.6682

C, H, Wd = 4, 4, 4
Mcorr = torch.tensor([[1.0, 0.9, 0.0, 0.0], [0.9, 1.0, 0.0, 0.0],
                      [0.0, 0.0, 1.0, -0.9], [0.0, 0.0, -0.9, 1.0]])
Lchol = torch.linalg.cholesky(Mcorr)
def sample_data(n):
    z = torch.randn(n, C, H, Wd)
    return torch.einsum('ij,njhw->nihw', Lchol, z)

class ActNorm(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.logs = nn.Parameter(torch.zeros(1, C, 1, 1)); self.b = nn.Parameter(torch.zeros(1, C, 1, 1))
        self.inited = False
    def forward(self, x):
        if not self.inited:
            with torch.no_grad():
                m = x.mean([0,2,3], keepdim=True); v = x.std([0,2,3], keepdim=True) + 1e-6
                self.b.data = -m/v; self.logs.data = -torch.log(v)
            self.inited = True
        return torch.exp(self.logs)*x + self.b, H*Wd*self.logs.sum()

class Inv1x1(nn.Module):
    def __init__(self, C):
        super().__init__(); Q, _ = torch.linalg.qr(torch.randn(C, C)); self.W = nn.Parameter(Q)
    def forward(self, x):
        return torch.einsum('ij,njhw->nihw', self.W, x), H*Wd*torch.slogdet(self.W)[1]

class AffineCoupling(nn.Module):
    def __init__(self, C, hid=32):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(C//2, hid, 3, padding=1), nn.ReLU(),
                                 nn.Conv2d(hid, hid, 1), nn.ReLU(), nn.Conv2d(hid, C, 3, padding=1))
        self.net[-1].weight.data.zero_(); self.net[-1].bias.data.zero_()
    def forward(self, x):
        xa, xb = x.chunk(2, 1); logs, t = self.net(xb).chunk(2, 1); logs = torch.tanh(logs)
        return torch.cat([torch.exp(logs)*xa + t, xb], 1), logs.sum([1,2,3])

class Glow(nn.Module):
    def __init__(self, C, K=4, use_1x1=True):
        super().__init__(); self.use_1x1 = use_1x1
        self.an = nn.ModuleList([ActNorm(C) for _ in range(K)])
        self.iv = nn.ModuleList([Inv1x1(C) for _ in range(K)])
        self.co = nn.ModuleList([AffineCoupling(C) for _ in range(K)])
    def forward(self, x):
        ld = 0
        for k in range(len(self.an)):
            x, l = self.an[k](x); ld = ld + l
            if self.use_1x1: x, l = self.iv[k](x); ld = ld + l
            x, l = self.co[k](x); ld = ld + l
        logpz = (-0.5*x**2 - 0.5*math.log(2*math.pi)).sum([1,2,3])
        return logpz + ld

def bpd(m, x): return (-m(x)/x[0].numel()/math.log(2)).mean()

def train(use_1x1=True, steps=2000):
    torch.manual_seed(0); m = Glow(C, 4, use_1x1); opt = torch.optim.Adam(m.parameters(), 1e-3); curve = []
    for i in range(steps):
        loss = -m(sample_data(256)).mean()/(C*H*Wd)
        opt.zero_grad(); loss.backward(); opt.step()
        if i % 100 == 0 or i == steps-1:
            with torch.no_grad(): curve.append((i, round(bpd(m, sample_data(2000)).item(), 4)))
    return curve

print("curve_on  =", train(use_1x1=True))
print("curve_off =", train(use_1x1=False))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A 1×1 convolution layer has channel-mixing matrix $W=\begin{pmatrix}1&2\\0&4\end{pmatrix}$ acting on a feature map of size $h=2$, $w=3$. What is its log-determinant (Eq. 9)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute $\det W$ for the triangular matrix: product of the diagonal, $1\times 4 = 4$. — _For a triangular matrix the determinant is the product of the diagonal entries._
- Take the per-pixel log: $\log 4 \approx 1.386294$. — _Eq. 9 needs $\log|\det W|$ per pixel._
- Multiply by $h\cdot w = 2\times 3 = 6$. — _The same $W$ acts at all 6 pixels; the Jacobian is block-diagonal with 6 identical blocks, so logs add 6 times._

**Answer:** $h\cdot w\cdot\log|\det W| = 6\times\log 4 \approx 6\times 1.386294 = 8.3178$.

</details>

**Problem 2.** Actnorm uses per-channel scale $s=(4,\,0.25)$ on a feature map with $h=w=2$. What is its log-determinant, and what does the value tell you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Sum the per-channel logs: $\log 4 + \log 0.25 = 1.386294 + (-1.386294) = 0$. — _Actnorm's log-det is $h\cdot w\cdot\sum\log|s|$ (Table 1)._
- Multiply by $h\cdot w = 4$: $4\times 0 = 0$. — _Same per-pixel repetition as the 1×1 conv._

**Answer:** Log-det $= 0$. Because $4\times 0.25 = 1$, this actnorm preserves total volume — it stretches one channel and compresses the other by the exact reciprocal.

</details>

**Problem 3.** ABLATION. In the notebook, deleting the invertible 1×1 convolution from every step makes the held-out bits/dimension go UP (worse). Explain why, mechanistically.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that affine coupling leaves one half of the channels ($y_b = x_b$) unchanged each step. — _Coupling can only transform variables conditioned on the untouched half._
- Note that, without the 1×1 conv (or a permutation) between steps, it is always the SAME half that is left unchanged. — _The 1×1 conv is what re-mixes channels so a different combination is transformed next step._
- Conclude the model can never apply a learned transform to the always-passed-through channel, so it cannot capture the correlation between the two channels in the toy data. — _Our toy data has strongly correlated channels; fitting them requires mixing._

**Answer:** Without the 1×1 conv the channels never mix, so one channel stays effectively a plain Gaussian and the model under-fits the channel correlation — the exact-likelihood objective reports this as higher (worse) bits/dimension. This is the paper's motivation for the learned 1×1 convolution.

</details>